# Домашнее задание: эмбеддинги слов и поисковая система для вопросов

В общих чертах вам предстоит:
- Обучить модель FastText на романе Льва Толстого «Война и мир».
- Дообучить полученные эмбеддинги на вопросах Quora, чтобы учесть новые токены и изменения значений слов со временем.
- Получить эмбеддинги имеющихся вопросов Quora и сохранить их для дальнейшего использования.
- Реализовать поиск наиболее близкого вопроса Quora для нового текста, которого модель ранее не видела.

Удачи!

# Обучение исходных эмбеддингов (4 балла)

Ваша первая задача — обучить эмбеддинги на романе Льва Толстого «Война и мир» с помощью FastText — расширения word2vec.

У FastText есть отличия от обычного word2vec; прочитать о них можно, например, [здесь](https://github.com/facebookresearch/fastText?tab=readme-ov-file). Для начала FastText удобно представлять как усовершенствованный word2vec.

Пожалуй, главное отличие заключается в том, что модель использует не только токены целых слов, но и их символьные n-граммы. Например, для слова "peace" представление с 2-граммами можно схематично записать как (peace, pe, ea, ac, ce).

Это позволяет модели обрабатывать слова вне словаря, комбинируя соответствующие подсловные компоненты. Кроме того, в некоторых случаях можно обойтись без нормализации слов (стемминга и лемматизации), сохранив семантическую информацию в языках с богатой морфологией.

In [ ]:
!pip install unidecode
!pip install faiss-cpu

import re
import string
from typing import Union, List, Tuple, Callable

import nltk
import faiss
import numpy as np
from unidecode import unidecode
from gensim.models import FastText
from gensim.models.word2vec import PathLineSentences

nltk.download('punkt_tab')

# Задаём seed для воспроизводимости
# Для полной детерминированности вычислений
# необходимо выполнять код в одном потоке
seed = 42

Загрузите текстовый файл романа «Война и мир» из репозитория проекта Gutenberg.

In [ ]:
!wget "https://www.gutenberg.org/files/2600/2600-0.txt" -O war_and_peace_raw.txt

--2024-11-21 01:23:22--  https://www.gutenberg.org/files/2600/2600-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3359405 (3.2M) [text/plain]
Saving to: ‘war_and_peace_raw.txt’

war_and_peace_raw.t 100%[===================>]   3.20M  12.2MB/s    in 0.3s    

2024-11-21 01:23:22 (12.2 MB/s) - ‘war_and_peace_raw.txt’ saved [3359405/3359405]



In [ ]:
wap_raw_file_path = 'war_and_peace_raw.txt'
wap_cleaned_file_path = 'war_and_peace_cleaned.txt'

В тексте есть информация, не нужная для нашей задачи, поэтому сначала необходимо предобработать файл.

In [ ]:
!head war_and_peace_raw.txt

The Project Gutenberg eBook of War and Peace, by Leo Tolstoy

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at
www.gutenberg.org. If you are not located in the United States, you
will have to check the laws of the country where you are located before
using this eBook.



In [ ]:
# Читаем текст из файла
with open(wap_raw_file_path, 'r', encoding='utf-8') as file:
    text = file.read()

# Удаляем служебные блоки в начале и конце текста
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK WAR AND PEACE ***"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK WAR AND PEACE ***"
text = text.split(start_marker, 1)[-1]  # Удаляем всё до маркера начала
text = text.split(end_marker, 1)[0]     # Удаляем всё после маркера конца

# Разбиваем текст на строки и отфильтровываем ненужные
lines = text.split("\n")
filtered_lines = []
for line in lines:
    line = line.strip()
    # Пропускаем пустые строки, заголовки глав и служебную информацию
    if not line or line.startswith("CHAPTER") or line.isupper():
        continue
    filtered_lines.append(line)

# Снова объединяем строки в один текст
cleaned_text = "\n".join(filtered_lines[2:])

По историческим причинам в тексте встречаются символы, не входящие в ASCII. Один из способов обработки — заменить их ближайшими аналогами из ASCII.

Учтите, что это не всегда необходимо и в некоторых случаях может ухудшить качество модели.

*Если вы считаете, что эти символы важны, оставьте их без изменений.*

In [ ]:
# Приводим символы к ASCII
cleaned_text_ascii = unidecode(cleaned_text)

Также важно понимать устройство данных и то, как модель обучает свои веса.

В нашем случае word2vec (FastText) обрабатывает текст построчно, перемещая по строке окно заданного размера, которое определяет контекст. Однако текст «Войны и мира» из Gutenberg разбит на строки довольно произвольно. Слова в одной строке могут относиться к разным предложениям, и тогда контекстное окно ошибочно объединит их в общий контекст.

Один из способов исправить это — сначала разбить текст на предложения. К счастью, весь текст романа помещается в оперативную память.

In [ ]:
# Разбиваем на предложения
sentences = nltk.sent_tokenize(cleaned_text_ascii)

sentences[:10]

['"Well, Prince, so Genoa and Lucca are now just family estates of the\nBuonapartes.',
 "But I warn you, if you don't tell me that this means war,\nif you still try to defend the infamies and horrors perpetrated by that\nAntichrist--I really believe he is Antichrist--I will have nothing\nmore to do with you and you are no longer my friend, no longer my\n'faithful slave,' as you call yourself!",
 'But how do you do?',
 'I see I\nhave frightened you--sit down and tell me all the news."',
 'It was in July, 1805, and the speaker was the well-known Anna Pavlovna\nScherer, maid of honor and favorite of the Empress Marya Fedorovna.',
 'With these words she greeted Prince Vasili Kuragin, a man of high\nrank and importance, who was the first to arrive at her reception.',
 'Anna\nPavlovna had had a cough for some days.',
 'She was, as she said, suffering\nfrom la grippe; grippe being then a new word in St. Petersburg, used\nonly by the elite.',
 'All her invitations without exception, written in

Как видите, в предложениях много нежелательных символов: произвольные переводы строк (`\n`), двойные дефисы (`--`) и т. п.

Для нашей задачи — обучения эмбеддингов FastText — имеет смысл предобработать текст: удалить пунктуацию и лишние символы, привести все слова к нижнему регистру. Можно также применить стемминг или лемматизацию, но FastText позволяет обойтись без этого и сохранить часть морфологической информации.

In [ ]:
# <<IMPORT YOUR FUNC HERE (1.1)>> — импортируйте свою функцию здесь
from solution import clean_wap_sentence

In [ ]:
# Предобрабатываем каждое предложение
processed_sentences = []

for sentence in sentences:
    <YOUR_CODE_HERE>

# Разделяем обработанные предложения переводами строк в соответствии с форматом Gensim
final_text = "\n".join(processed_sentences)

with open(wap_cleaned_file_path, 'w') as outfile:
    outfile.write(final_text)

# Сохраняем число предложений для настройки параметров модели Gensim
wap_sentences_count = len(processed_sentences)

In [ ]:
# Посмотрим на промежуточный результат
!sed -n 1,4p war_and_peace_cleaned.txt

well prince so genoa and lucca are now just family estates of the buonapartes
but i warn you if you don't tell me that this means war if you still try to defend the infamies and horrors perpetrated by that antichrist i really believe he is antichrist i will have nothing more to do with you and you are no longer my friend no longer my 'faithful slave' as you call yourself
but how do you do
i see i have frightened you sit down and tell me all the news


Данные подготовлены — наконец можно обучить модель!

В этом задании мы используем [реализацию FastText из Gensim](https://radimrehurek.com/gensim/models/fasttext.html). У неё есть небольшие накладные расходы по сравнению с [оригинальной реализацией](https://fasttext.cc/), зато удобный API, который также поддерживает дообучение существующих моделей.

**Внимательно прочитайте рекомендации ниже: они помогут пройти проверки.**

0. Задайте скорость обучения `3e-2`. FastText в Gensim будет линейно уменьшать её по мере обучения.
1. Вы можете экспериментировать с размерностью векторов, но она существенно влияет на качество и скорость работы. При размерности меньше 64 эмбеддинги могут плохо передавать значения слов. Размерность больше 300 замедлит работу и, скорее всего, не даст соразмерного выигрыша. Рекомендуем 100 как разумный компромисс для прохождения проверки.
2. Обязательно используйте режим skip-gram. Обычно он даёт более качественные семантические представления, хотя обучается сравнительно медленно.
3. Для имеющегося объёма данных, скорее всего, достаточно 5 эпох, но вы можете экспериментировать.
4. Не забудьте задать `seed`. Однако при использовании нескольких потоков одного этого, скорее всего, будет недостаточно для полной детерминированности.
5. Размер окна — важный параметр. Чем шире окно, тем более широкий контекст предложения учитывает эмбеддинг, но тем слабее может отражаться собственная семантическая специфика слова. Экспериментируйте. Окно размером 5 — хорошая отправная точка для прохождения блока проверок `assert`.
6. При инициализации модели FastText задайте `bucket = 100_000`.

Обратите внимание: мы не делим данные на обучающую и тестовую выборки. В этой задаче окончательную оценку даёте вы. Если эмбеддинги кажутся вам осмысленными и проходят проверки, такой результат нас устраивает.

In [ ]:
# <<IMPORT YOUR FUNC HERE (1.2)>> — импортируйте свою функцию здесь
from solution import (
    build_fasttext_model, 
    train_fasttext_model
    )

In [ ]:
# Используем PathLineSentences для потокового чтения данных из файла
wap_sentences = PathLineSentences(wap_cleaned_file_path)

# Создаём модель FastText с помощью Gensim (build_fasttext_model)
model = <YOUR_CODE_HERE>

# Обучаем модель FastText (train_fasttext_model)
model = <YOUR_CODE_HERE>

model.save("solution_data/wap_fasttext_model.bin")

Ниже можно поэкспериментировать с обученной моделью.

In [ ]:
print(*model.wv.most_similar('peace', topn=10), sep='\n')

('endure', 0.8580781817436218)
('failure', 0.8559691905975342)
('source', 0.8493669629096985)
('freedom', 0.8472211360931396)
('possess', 0.8464798927307129)
('arise', 0.8428733944892883)
('encourage', 0.8408569693565369)
('hope', 0.8401851058006287)
('success', 0.839286744594574)
('faith', 0.8383597731590271)


FastText позволяет строить представления слов вне словаря с помощью символьных n-грамм…

In [ ]:
'computation' in model.wv.key_to_index

False

…хотя такие слова могут быть весьма далеки от контекстов, на которых обучалась модель.

In [ ]:
print(*model.wv.most_similar('computation', topn=10), sep='\n')

('consultation', 0.9750005006790161)
('combination', 0.9707764387130737)
('deputation', 0.9693651795387268)
('subordination', 0.9676422476768494)
('manifestation', 0.9660473465919495)
('confirmation', 0.9654446244239807)
('participation', 0.9653453826904297)
('population', 0.9614249467849731)
('gravitation', 0.9605057239532471)
('situation', 0.9602699279785156)


In [ ]:
# Блок отладки
model_vocabulary = set(model.wv.key_to_index.keys())
most_similar_to_peace = list(zip(*model.wv.most_similar('peace', topn=50)))[0]

assert model.vector_size >= 64 and model.vector_size <= 300, 'Please check your embedding size.'
assert model.sg == 1, 'Please use skip-gram method for consistency. Also, despite being faster, CBOW usually generates inferior word embeddings for rare words.'
assert model.alpha == 3e-2, 'It is expected that you overwrite the default alpha for this task.'
assert len(model_vocabulary) > 5500 and len(model_vocabulary) < 7500, 'There is something wrong with your tokenization. Check the pipeline and use a default *ngram=1*'
assert 'religion' in most_similar_to_peace, 'Embeddings look odd. Make sure to follow instructions!'
assert model.wv.bucket == 100_000, 'Set the value to 100_000 so that the model doesn’t run out of memory, while still having sufficient potential for training!'

print('Congrats!')

Congrats!


# Дообучение на датасете Quora (3 балла)

Теперь у нас есть эмбеддинги, отражающие семантическое пространство мира Льва Толстого. Перейдём к более современным представлениям.

Одна из проблем — в словаре модели нет многих современных слов. Хотя FastText позволяет построить их представления из n-грамм, эти представления не учитывают контексты употребления таких слов.
Другая проблема — контексты употребления некоторых слов могли измениться с течением времени.

Поэтому наша задача — обогатить обученные векторы новыми данными, сохранив ранее полученную информацию.

In [ ]:
# Загружаем данные
!wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora_raw.txt
# Альтернативная ссылка для загрузки: https://yadi.sk/i/BPQrUu1NaTduEw

--2024-11-21 01:24:17--  https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1 [following]
--2024-11-21 01:24:18--  https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucd5063bb481f8b4f384c225fd93.dl.dropboxusercontent.com/cd/0/inline/CexfgI0XVqjGK3m-ZDd3DnDKA2-pST3dRzYpdPlOIgPbe5Rpf9Ag5QoBJWy4fwPgrDrX2fdtvx5v5FEd6koxzj3VDzDPEWWQguiJXfL70oPUkkq_nJYIXiodgRxKxEDLMPU/file?dl=1# [following]
--2024-11-21 01:24:18--  https://ucd5063bb481f8b4f384c225fd93.dl.dropboxusercontent.com/cd/0/inline/CexfgI0XVqj

In [ ]:
# Посмотрим на первые строки датасета Quora
!sed -n 1,5p quora_raw.txt

Can I get back with my ex even though she is pregnant with another guy's baby?
What are some ways to overcome a fast food addiction?
Who were the great Chinese soldiers and leaders who fought in WW2?
What are ZIP codes in the Bay Area?
Why was George RR Martin critical of JK Rowling after losing the Hugo award?


Как и раньше, нужно привести данные к единому виду для обучения.

К счастью, здесь каждая строка уже содержит целое предложение — и даже в ASCII!

In [ ]:
# <<IMPORT YOUR FUNC HERE (2.1)>> — импортируйте свою функцию здесь
from solution import preprocess_file

In [ ]:
# Пути к входному и выходному файлам
quora_raw_file_path = 'quora_raw.txt'
quora_processed_file_path = 'quora_processed.txt'

# Предобрабатываем файл и подсчитываем число обработанных строк
quora_sentences_count = preprocess_file(quora_raw_file_path, quora_processed_file_path)

print(f'Lines processed: {quora_sentences_count}')

Lines processed: 537272


Тонкость в том, что мы продолжаем обучение на более крупном датасете с заметно другой тематикой. Чтобы не потерять полностью семантику, усвоенную по «Войне и миру», разумно продолжить с той скорости обучения, на которой завершился предыдущий этап, и/или не задавать слишком много эпох.

In [ ]:
# Как и раньше, используем средства Gensim для перебора строк
quora_sentences = PathLineSentences(quora_processed_file_path)

# Загружаем модель для дообучения
finetuned_model = FastText.load("solution_data/wap_fasttext_model.bin")

In [ ]:
# <<IMPORT YOUR FUNC HERE (2.2)>> — импортируйте свою функцию здесь
from solution import (
    update_fasttext_vocab,
    update_fasttext_model
    )

In [ ]:
# Обновляем словарь на основе новых предложений
<YOUR_CODE_HERE>

# Продолжаем обучение модели на новых данных
# Не забудьте правильно передать параметры
<YOUR_CODE_HERE>

finetuned_model.save("solution_data/wap_quora_fasttext_model.bin")

In [ ]:
finetuned_model_vocabulary = set(finetuned_model.wv.key_to_index.keys())
new_words = finetuned_model_vocabulary - model_vocabulary

print("Number of new words added:", len(new_words))
print("New words:", list(new_words)[10:20])

Number of new words added: 23878
New words: ['tripadvisor', 'bowie', 'hrd', 'sasikala', 'traits', 'boss', 'yrs', 'popped', 'appstore', 'diver']


In [ ]:
def print_comparison(
    model1: Union[FastText, None],
    model2: Union[FastText, None],
    word: str = 'peace',
    top_n: int = 10
) -> None:
    """
    Сравнивает top-N наиболее похожих слов для заданного слова в двух моделях FastText.
    Выводит таблицу для наглядной оценки изменений после дообучения.

    Аргументы:
        model1 (FastText | None): Первая модель FastText (исходная, до дообучения).
        model2 (FastText | None): Вторая модель FastText (после дообучения).
        word (str, необязательный): Слово, для которого ищутся похожие слова.
        top_n (int, необязательный): Число наиболее похожих слов для сравнения.

    Возвращает:
        None: Результаты выводятся непосредственно в консоль.
    """
    q0 = model1.wv.most_similar(word, topn=top_n)
    q1 = model2.wv.most_similar(word, topn=top_n)

    # Выводим заголовок
    print(f'Query: {word}\n')
    print(f"{'pos':<5} {'model_1':<15} {'score_1':<10} {'model_2':<15} {'score_2':<10}")
    print("-" * (5 + 15 + 10 + 15 + 10))

    # Выводим каждую из top_n записей
    for pos, (word0, score0), (word1, score1) in zip(range(top_n), q0, q1):
        # Выводим оценки с четырьмя знаками после запятой
        print(f"{pos:<5} {word0:<15} {score0:.4f}     {word1:<15} {score1:.4f}")


In [ ]:
# Вызываем функцию для вывода результатов
print_comparison(model, finetuned_model, word='war', top_n=10)

Query: war

pos   model_1         score_1    model_2         score_2   
-------------------------------------------------------
0     campaign        0.8246     communication   0.9170
1     communication   0.8218     warg            0.9153
2     b               0.8195     abolition       0.9105
3     consideration   0.8105     telecommunication 0.9095
4     campaigns       0.8071     revolution      0.9091
5     description     0.8068     consideration   0.9075
6     revolution      0.8060     alliteration    0.9069
7     affair          0.7937     capitalization  0.9060
8     execution       0.7928     ministry        0.9059
9     vital           0.7927     adulteration    0.9054


In [ ]:
# Блок проверок assert для отладки и тестирования
assert finetuned_model.alpha <= model.min_alpha, 'setting a learning rate too high will result in model forgetting previous information, especially when finetuning on a larger dataset!'
assert len(new_words) > 20_000 and len(new_words) < 30_000, 'There is likely something wrong with your preprocessing pipeline. There must be more new words after finetuning!'
assert 'religion' in list(zip(*finetuned_model.wv.most_similar('peace', topn=50)))[0], 'Embeddings changed dramatically! Consider setting lower start_apha and fewer epochs.'
assert 'evolution' in list(zip(*finetuned_model.wv.most_similar('war', topn=50)))[0], 'Embeddings didn\'t change as was expected! Consider setting higher start_apha and more epochs, check *total_examples* param.'

print('Gratz!')

Gratz!


На практике полезно экспериментировать с разными стратегиями дообучения: менять параметр `alpha`, увеличивать число эпох или даже пробовать «замораживать» некоторые эмбеддинги, обновляя только определённые части словаря.

# Поиск похожих вопросов Quora по новому запросу (3 балла)

## Прямой поиск с помощью массива NumPy

Для семантического поиска нам нужно уметь представлять новое предложение одним вектором. Самый простой способ — усреднить эмбеддинги отдельных слов.

Можно также попробовать разные способы агрегации (пулинга: min, max, mean, softmax) и [TF-IDF](https://www.geeksforgeeks.org/understanding-tf-idf-term-frequency-inverse-document-frequency/). Для TF-IDF нужно заранее вычислить IDF по имеющемуся корпусу, а затем умножать его на TF соответствующего слова в конкретном предложении.

Далее предполагается использовать косинусное сходство для сравнения векторов. Поэтому полезно нормализовать все векторы перед сохранением: для нормализованных векторов вычисление косинусного сходства сводится к скалярному произведению, что ускоряет поиск:
$$
cos(\phi) = \frac{<u, v>}{|u||v|}.
$$

In [ ]:
# <<IMPORT YOUR FUNC HERE (3.1)>> — импортируйте свою функцию здесь
from solution import get_text_embedding, normalize_vector

In [ ]:
# Блок отладки и тестирования
question = "Love peace"
text_embedding = get_text_embedding(question, finetuned_model)
dummy_word_embedding = (finetuned_model.wv['love'] + finetuned_model.wv['peace']) / 2

assert text_embedding.shape[0] == finetuned_model.wv.vector_size, "Check axis along which values are aggregated"
assert all(normalize_vector(dummy_word_embedding) == text_embedding), "You need to return a normalized mean embedding for now, nothing fancy!"

print('Good job!')

Good job!


Наконец, реализуем поиск похожих предложений.

Вам нужно получить эмбеддинг каждого вопроса Quora и сохранить эти векторы в форме, удобной для поиска.

Чтобы получилась поисковая система, новый пользовательский запрос тоже нужно преобразовать в эмбеддинг, а затем выполнить поиск по имеющемуся хранилищу векторов.

Начнём с простого подхода: создадим массив NumPy для хранения эмбеддингов имеющихся вопросов Quora.

Обязательно применяйте ту же предобработку, что и при обучении модели!

In [ ]:
# <<IMPORT YOUR FUNC HERE (3.2 3.3)>> — импортируйте свою функцию здесь
from solution import create_embeddings_storage, find_closest_match_np

In [ ]:
# Создаём массив для хранения нормализованных эмбеддингов
embeddings_storage_np = create_embeddings_storage("quora_processed.txt", finetuned_model)

In [ ]:
with open(quora_processed_file_path, 'r') as f:
    for line in f:
        test_embedding = get_text_embedding(line, finetuned_model)
        break

assert embeddings_storage_np.shape == (quora_sentences_count, finetuned_model.vector_size), "The vector storage must be of size (num_quora_sentences, embedding_dim)"
assert embeddings_storage_np[0].mean() == test_embedding.mean(), 'Embedding of the first quora question does not correspond to the first enty in the storage!'
print('Looking good so far!')

Looking good so far!


Пока будем хранить все вопросы Quora в оперативной памяти в виде списка Python.

На практике можно рассмотреть отдельную базу данных с получением записи по индексу за константное время.

In [ ]:
# Сохраняем исходные вопросы для проверки результатов вручную
quora_questions_list = []
with open(quora_raw_file_path, 'r') as file:
    for line in file:
        quora_questions_list.append(line.strip())

print(f"Total questions processed: {len(quora_questions_list)}")

Total questions processed: 537272


In [ ]:
def fetch_and_display_closest_match(query_function: Callable[..., Tuple[List[int], List[float]]], **kwargs) -> Tuple[List[int], List[float]]:
    """
    Получает и выводит наиболее близкие вопросы из датасета Quora для заданного запроса.

    С помощью переданной функции поиска находит top-k наиболее похожих вопросов
    и выводит их вместе с оценками сходства.

    Аргументы:
        query_function (Callable): Функция, принимающая параметры поиска и возвращающая
            индексы top-k наиболее похожих вопросов и оценки их сходства
            в виде кортежа (List[int], List[float]).
        **kwargs: Дополнительные аргументы для query_function, включая текст запроса.

    Возвращает:
        Tuple[List[int], List[float]]: Кортеж из:
            - Списка индексов top-k наиболее похожих вопросов в датасете.
            - Списка оценок сходства для этих вопросов.
    """

    # Получаем индексы top-k результатов и оценки сходства из функции поиска
    top_k_indices, top_k_similarities = query_function(**kwargs)

    # Получаем тексты вопросов по найденным индексам
    top_k_questions = [quora_questions_list[i] for i in top_k_indices]

    # Выводим запрос и top-k результатов с оценками сходства
    print(f"Query: {kwargs['query']}")
    print("\nTop Matches:")
    for i, (question, similarity) in enumerate(zip(top_k_questions, top_k_similarities), 1):
        print(f"{i}. {question} (Similarity: {similarity:.4f})")

    return top_k_indices, top_k_similarities


In [ ]:
# Блок тестирования
new_question = "How can I find inner peace?"

top_k_indices, top_k_similarities = fetch_and_display_closest_match(query_function=find_closest_match_np,
                                                                    query=new_question, model=finetuned_model,
                                                                    embeddings_storage=embeddings_storage_np,
                                                                    k=10)

assert 446084 in top_k_indices, 'Your embeddings look odd. = ('

Query: How can I find inner peace?

Top Matches:
1. How can I create inner peace? (Similarity: 0.9925)
2. How can I find a career mentor? (Similarity: 0.9918)
3. How can I find passion? (Similarity: 0.9918)
4. How can I find a successful mentor? (Similarity: 0.9912)
5. How can I find a programmer mentor? (Similarity: 0.9904)
6. How can I find inspiration? (Similarity: 0.9898)
7. How can I prevent business failure? (Similarity: 0.9891)
8. How can I find app developers? (Similarity: 0.9889)
9. How can I find a JavaScript programmer mentor? (Similarity: 0.9888)
10. How can I find a startup mentor? (Similarity: 0.9887)


Поэкспериментируйте со своей новой поисковой системой!

In [ ]:
new_question = "What is the future of artificial intelligence?"

_, _ = fetch_and_display_closest_match(query_function=find_closest_match_np,
                                       query=new_question, model=finetuned_model,
                                       embeddings_storage=embeddings_storage_np,
                                       k=10)


Query: What is the future of artificial intelligence?

Top Matches:
1. What is the future of artificial intelligence? (Similarity: 1.0000)
2. What is the scope of Artificial Intelligence? (Similarity: 0.9976)
3. What is the future of Social Media? (Similarity: 0.9968)
4. What is the future of Pharmaceutical industry? (Similarity: 0.9968)
5. What is the future of Chinese economy? (Similarity: 0.9967)
6. What is the future of internet piracy? (Similarity: 0.9967)
7. What is the indefinite integral of [math]\sin (x^3)[/math] ? (Similarity: 0.9966)
8. What is the future of virtual reality? (Similarity: 0.9964)
9. What is the nature of public administration? (Similarity: 0.9963)
10. What is the future of chemical engineering? (Similarity: 0.9963)


In [ ]:
import time

iterations = 100

start_time = time.time()

for _ in range(iterations):
    _, _ = find_closest_match_np('hello there', finetuned_model, embeddings_storage_np, k=10)

time_elapsed = time.time() - start_time
average_time = time_elapsed / iterations

print(f'Time elapsed: {time_elapsed:.4f} seconds.')
print(f'Average time: {average_time:.4f} seconds.')

Time elapsed: 12.0655 seconds.
Average time: 0.1207 seconds.


Обратите внимание: на практике результаты частых запросов кешируют.

# Поиск с помощью векторной базы данных

Однако зачастую данных слишком много, чтобы они поместились в оперативную память (RAM), тем более в память видеокарты (VRAM). Поэтому нужен подход, более подходящий для таких объёмов, чем поиск по массиву NumPy. Кроме того, мы не можем позволить себе тратить O(n) времени на каждый запрос.

Векторные базы данных — базы, специально предназначенные для хранения векторов и быстрого поиска, — решают проблему скорости, но в описываемом подходе хранилище всё равно приходится держать в RAM. На практике разные части данных могут находиться на нескольких связанных между собой машинах (шардах).

Ниже уже реализован один из способов построения хранилища эмбеддингов. Внимательно изучите код.

Подробнее можно прочитать о библиотеке [FAISS](https://faiss.ai/index.html) и алгоритме [HNSW](https://arxiv.org/abs/1603.09320) — эффективном приближённом поиске k ближайших соседей.

In [ ]:
# <<IMPORT YOUR FUNC HERE (7)>> — импортируйте свою функцию здесь
from solution import preprocess_line

In [ ]:
def build_faiss_hnsw_index(dimension: int, ef_construction: int = 200, M: int = 32) -> faiss.IndexHNSWFlat:
    """
    Создаёт индекс FAISS HNSW для поиска по косинусному сходству.

    Инициализирует индекс HNSW (Hierarchical Navigable Small World) для эффективного
    приближённого поиска ближайших соседей по косинусному сходству
    с использованием нормализованных эмбеддингов модели FastText.

    Параметры:
        dimension (int): Размерность эмбеддингов (длина каждого вектора).
        ef_construction (int, необязательный): Параметр компромисса между скоростью
            построения индекса и точностью. По умолчанию 200.
        M (int, необязательный): Число соседей в графе; определяет компромисс
            между расходом памяти и точностью. По умолчанию 32.

    Возвращает:
        index (faiss.IndexHNSWFlat): Инициализированный индекс FAISS HNSW.
    """
    index = faiss.IndexHNSWFlat(dimension, M)  # Индекс HNSW
    index.hnsw.efConstruction = ef_construction  # Точность построения индекса
    index.metric_type = faiss.METRIC_INNER_PRODUCT  # Косинусное сходство через скалярное произведение нормализованных векторов
    return index


def populate_faiss_index(index: faiss.Index, model, dataset_path: str, batch_size: int = 10000):
    """
    Заполняет индекс FAISS HNSW нормализованными эмбеддингами из датасета.

    Читает датасет построчно, предобрабатывает каждый вопрос, вычисляет его эмбеддинг
    и добавляет эмбеддинги в индекс FAISS пакетами.

    Параметры:
        index (faiss.Index): Индекс FAISS, который нужно заполнить.
        model: Обученная модель FastText для получения эмбеддингов.
        dataset_path (str): Путь к файлу датасета (один вопрос на строку).
        batch_size (int, необязательный): Число вопросов в одном пакете. По умолчанию 10000.
    """
    with open(dataset_path, "r") as f:
        buffer = []
        for line in f:
            # Предобрабатываем строку и получаем её эмбеддинг
            question = preprocess_line(line.strip())
            embedding = get_text_embedding(question, model)
            buffer.append(embedding)

            # Добавляем эмбеддинги в индекс пакетами
            if len(buffer) >= batch_size:
                index.add(np.array(buffer, dtype=np.float32))
                buffer = []  # Очищаем буфер после добавления

        # После завершения цикла добавляем оставшиеся эмбеддинги, если они есть
        if buffer:
            index.add(np.array(buffer, dtype=np.float32))


def search_faiss_index(embeddings_storage: faiss.Index, query: str, model, k: int = 5) -> Tuple[List[int], List[float]]:
    """
    Ищет в индексе FAISS наиболее близкие соответствия запросу.

    Вычисляет эмбеддинг запроса, находит наиболее похожие вопросы в индексе FAISS
    и возвращает индексы и оценки косинусного сходства top-k результатов.

    Параметры:
        embeddings_storage (faiss.Index): Индекс FAISS для поиска.
        query (str): Строка поискового запроса.
        model: Обученная модель FastText для получения эмбеддингов.
        k (int, необязательный): Число ближайших соответствий. По умолчанию 5.

    Возвращает:
        Tuple[List[int], List[float]]:
            - List[int]: Индексы top-k наиболее похожих вопросов.
            - List[float]: Оценки косинусного сходства top-k результатов
              (в исходном коде переменные названы distances).
    """
    # Получаем нормализованный эмбеддинг запроса с учётом предобработки
    query_embedding = get_text_embedding(query, model)

    # Выполняем поиск в embeddings_storage (индексе FAISS)
    top_k_distances, top_k_indices = embeddings_storage.search(np.array([query_embedding], dtype=np.float32), k)

    # Приводим результат к формату поиска по хранилищу NumPy
    top_k_indices_list = top_k_indices[0].tolist()
    top_k_distances_list = top_k_distances[0].tolist()

    return top_k_indices_list, top_k_distances_list


Построение индекса FAISS может занять некоторое время — это нормально. В некотором смысле мы тратим вычислительные ресурсы при построении индекса, чтобы затем быстрее обрабатывать запросы.

In [ ]:
# Задаём размерность векторов эмбеддингов
embedding_dimension = finetuned_model.vector_size  # Зависит от модели FastText

# Строим индекс HNSW
hnsw_index = build_faiss_hnsw_index(embedding_dimension)

# Заполняем индекс данными из quora_processed.txt
populate_faiss_index(hnsw_index, finetuned_model, "quora_processed.txt")


В исходном пояснении говорится, что индекс FAISS построен для евклидова расстояния, а не непосредственно для косинусного сходства. Для нормализованных векторов эти величины связаны формулой:

$$
cosine \space similarity = 1 - \frac{euclidian \space distance^2}{2}
$$

> **Примечание:** в оригинале есть сознательное расхождение между текстом и кодом: функция `build_faiss_hnsw_index` явно задаёт `faiss.METRIC_INNER_PRODUCT`. При нормализованных векторах это скалярное произведение, равное косинусному сходству. Формула выше верна для векторов единичной длины; дополнительно применять её к уже вычисленному косинусному сходству не нужно.

Результаты могут не полностью совпадать с результатами поиска по массиву NumPy, поскольку HNSW выполняет приближённый поиск.

In [ ]:
# Пример запроса
query_text = "How can I find inner peace?"

top_k_indices, top_k_similarities = fetch_and_display_closest_match(query_function=search_faiss_index,
                                                                    query=query_text, model=finetuned_model,
                                                                    embeddings_storage=hnsw_index,
                                                                    k=10)

Query: How can I find inner peace?

Top Matches:
1. How can I find a career mentor? (Similarity: -0.0164)
2. How can I find passion? (Similarity: -0.0165)
3. How can I find a successful mentor? (Similarity: -0.0177)
4. How can I find a programmer mentor? (Similarity: -0.0193)
5. How can I find inspiration? (Similarity: -0.0205)
6. How can I prevent business failure? (Similarity: -0.0218)
7. How can I find app developers? (Similarity: -0.0222)
8. How can I find a JavaScript programmer mentor? (Similarity: -0.0223)
9. How can I find a startup mentor? (Similarity: -0.0225)
10. How can I find a perfect co-founder? (Similarity: -0.0228)


In [ ]:
import time

iterations = 100

start_time = time.time()

for _ in range(iterations):
    _, _ = search_faiss_index(hnsw_index, query_text, finetuned_model, k=10)

time_elapsed = time.time() - start_time
average_time = time_elapsed / iterations

print(f'Time elapsed: {time_elapsed:.4f} seconds.')
print(f'Average time: {average_time:.4f} seconds.')

Time elapsed: 0.0199 seconds.
Average time: 0.0002 seconds.


Обратите внимание: мы добились ускорения на два порядка!

> **Примечание:** это наблюдение из оригинала. Фактическое ускорение в вашем запуске зависит от оборудования, данных и настроек; сравните измеренное среднее время двух методов.

Что можно сделать дальше?

В части эмбеддингов можно улучшить способ получения вектора отдельного слова или ввести веса для эмбеддингов слов, например на основе TF-IDF. Далее в курсе вы познакомитесь с более качественными, но и существенно более вычислительно затратными способами представления произвольного текста, чем агрегация отдельных векторов. При ограниченных вычислительных ресурсах эмбеддинги слов всё ещё остаются вполне подходящим вариантом.

В части поисковой системы кеширование позволяет избежать повторного выполнения одинаковых запросов. Например, посмотрите на [Redis](https://redis.io/learn/howtos/solutions/microservices/caching).